## Mini Dataset Card (UCI HAR)

- **Motivation:** Build and benchmark smartphone-based human activity recognition models from wearable inertial sensor windows.
- **Target definition:** 6-way activity classification per window (WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING).
- **Data source/license:** UCI Human Activity Recognition Using Smartphones Dataset. Dataset README states AS-IS distribution and that commercial use is prohibited.
- **Signal description:** 9 inertial channels (body_acc_{x,y,z}, body_gyro_{x,y,z}, total_acc_{x,y,z}), sampled at 50 Hz, window length T=128 (2.56 s) with 50% overlap.
- **Limitations/risks:**
  - Small cohort (30 subjects, age 19-48) and single phone placement (waist), limiting generalization.
  - Overlapping windows may inflate apparent performance if split incorrectly.
  - Dataset is preprocessed/cleaned; deployment data can be noisier and shifted.
  - Potential subject/activity imbalance can bias metrics if not monitored.


In [15]:
from pathlib import Path
import numpy as np

DATA_ROOT = Path('UCI-HAR Dataset')
INERTIAL_CHANNELS = [
    'body_acc_x', 'body_acc_y', 'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z',
    'total_acc_x', 'total_acc_y', 'total_acc_z',
]

def load_activity_map(root: Path):
    names = [line.strip().split()[1] for line in (root / 'activity_labels.txt').read_text().splitlines() if line.strip()]
    ids = np.arange(1, len(names) + 1).tolist()
    return dict(zip(ids, names))

def load_split(split: str, root: Path = DATA_ROOT):
    split_dir = root / split
    sig_dir = split_dir / 'Inertial Signals'

    channel_arrays = []
    for ch in INERTIAL_CHANNELS:
        arr = np.loadtxt(sig_dir / f'{ch}_{split}.txt')
        channel_arrays.append(arr)

    X = np.stack(channel_arrays, axis=-1)
    y = np.loadtxt(split_dir / f'y_{split}.txt', dtype=int)
    subjects = np.loadtxt(split_dir / f'subject_{split}.txt', dtype=int)
    return X, y, subjects

X_train, y_train, subj_train = load_split('train')
X_test, y_test, subj_test = load_split('test')

X_all = np.concatenate([X_train, X_test], axis=0)
y_all = np.concatenate([y_train, y_test], axis=0)
subj_all = np.concatenate([subj_train, subj_test], axis=0)
activity_map = load_activity_map(DATA_ROOT)

print('Loaded data successfully.')


Loaded data successfully.


## Data Sanity Checks


In [17]:
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'X_all shape: {X_all.shape}')

label_ids, label_counts = np.unique(y_all, return_counts=True)
print('\nLabel distribution (all windows):')
for label_id, count in zip(label_ids, label_counts):
    print(f"  {label_id:>2} ({activity_map.get(int(label_id), 'UNKNOWN')}): {int(count)}")

nan_count = int(np.isnan(X_all).sum())
inf_count = int(np.isinf(X_all).sum())
print(f'\nNaN count in X_all: {nan_count}')
print(f'Inf count in X_all: {inf_count}')
assert nan_count == 0, 'NaN values detected in X_all'
assert inf_count == 0, 'Inf values detected in X_all'
print('All good !')


X_train shape: (7352, 128, 9)
X_test shape: (2947, 128, 9)
X_all shape: (10299, 128, 9)

Label distribution (all windows):
   1 (WALKING): 1722
   2 (WALKING_UPSTAIRS): 1544
   3 (WALKING_DOWNSTAIRS): 1406
   4 (SITTING): 1777
   5 (STANDING): 1906
   6 (LAYING): 1944

NaN count in X_all: 0
Inf count in X_all: 0
All good !


## Leakage Audit (Subject-Disjoint Split)


In [19]:
rng = np.random.default_rng(42)
unique_subjects = np.unique(subj_all)
shuffled_subjects = rng.permutation(unique_subjects)

n_train_subjects = int(0.7 * len(unique_subjects))
train_subjects = set(shuffled_subjects[:n_train_subjects].tolist())
val_subjects = set(shuffled_subjects[n_train_subjects:].tolist())

train_mask = np.isin(subj_all, list(train_subjects))
val_mask = np.isin(subj_all, list(val_subjects))

X_tr_subj, y_tr_subj, subj_tr_subj = X_all[train_mask], y_all[train_mask], subj_all[train_mask]
X_val_subj, y_val_subj, subj_val_subj = X_all[val_mask], y_all[val_mask], subj_all[val_mask]

subject_overlap = set(np.unique(subj_tr_subj)).intersection(set(np.unique(subj_val_subj)))
overlap = len(subject_overlap)

print('Subject-disjoint split shapes:')
print(f'Train: {X_tr_subj.shape}, subjects={len(np.unique(subj_tr_subj))}')
print(f'Val: {X_val_subj.shape}, subjects={len(np.unique(subj_val_subj))}')
print(f'Subject overlap count: {overlap}')
print(f'Subject overlap set: {sorted(subject_overlap)}')
assert overlap == 0, 'Leakage detected.'
print('Leakage audit passed: overlap = 0.')


Subject-disjoint split shapes:
Train: (7369, 128, 9), subjects=21
Val: (2930, 128, 9), subjects=9
Subject overlap count: 0
Subject overlap set: []
Leakage audit passed: overlap = 0.


### Baseline 0: Random Predictor

In [20]:
seed = 42
rng = np.random.default_rng(seed)

classes = np.unique(y_train)
y_pred_random = rng.choice(classes, size=len(y_test), replace=True)
random_acc = float((y_pred_random == y_test).mean())

print(f'Baseline 0 - Random predictor (seed={seed})')
print(f'Classes sampled from: {classes.tolist()}')
print(f'Test accuracy: {random_acc:.4f}')

pred_labels, pred_counts = np.unique(y_pred_random, return_counts=True)
print('Predicted label distribution:')
for label_id, count in zip(pred_labels, pred_counts):
    print(f"  {label_id:>2} ({activity_map.get(int(label_id), 'UNKNOWN')}): {int(count)}")


Baseline 0 - Random predictor (seed=42)
Classes sampled from: [1, 2, 3, 4, 5, 6]
Test accuracy: 0.1642
Predicted label distribution:
   1 (WALKING): 513
   2 (WALKING_UPSTAIRS): 478
   3 (WALKING_DOWNSTAIRS): 491
   4 (SITTING): 485
   5 (STANDING): 472
   6 (LAYING): 508


## Baseline 1 (Required): MLP on Flattened Windows


In [21]:
import torch
import torch.nn as nn

# subject-disjoint train/validation split using training subjects
split_seed = 42
rng = np.random.default_rng(split_seed)
train_subject_ids = np.unique(subj_train)
shuffled = rng.permutation(train_subject_ids)
n_val_subjects = max(1, int(0.2 * len(train_subject_ids)))
val_subjects = set(shuffled[:n_val_subjects].tolist())
fit_subjects = set(shuffled[n_val_subjects:].tolist())
fit_mask = np.isin(subj_train, list(fit_subjects))
val_mask = np.isin(subj_train, list(val_subjects))
X_fit = X_train[fit_mask]
y_fit = y_train[fit_mask]
X_val = X_train[val_mask]
y_val = y_train[val_mask]

# flattening windows
X_fit_flat = X_fit.reshape(X_fit.shape[0], -1).astype(np.float32)
X_val_flat = X_val.reshape(X_val.shape[0], -1).astype(np.float32)

# stndardizing for efficiency (using fit split stats only to avoid leakage)
mean = X_fit_flat.mean(axis=0, keepdims=True)
std = X_fit_flat.std(axis=0, keepdims=True) + 1e-6
X_fit_std = (X_fit_flat - mean) / std
X_val_std = (X_val_flat - mean) / std

# converting labels
y_fit_0 = (y_fit - 1).astype(np.int64)
y_val_0 = (y_val - 1).astype(np.int64)

# training mlp
torch.manual_seed(42)
device = torch.device('cpu')
X_fit_t = torch.from_numpy(X_fit_std).to(device)
y_fit_t = torch.from_numpy(y_fit_0).to(device)
X_val_t = torch.from_numpy(X_val_std).to(device)
y_val_t = torch.from_numpy(y_val_0).to(device)
model = nn.Sequential(
    nn.Linear(X_fit_t.shape[1], 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, len(np.unique(y_train))),
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
batch_size = 256
epochs = 15
n_fit = X_fit_t.shape[0]
for _ in range(epochs):
    perm = torch.randperm(n_fit, device=device)
    for i in range(0, n_fit, batch_size):
        idx = perm[i:i+batch_size]
        logits = model(X_fit_t[idx])
        loss = criterion(logits, y_fit_t[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
model.eval()
with torch.no_grad():
    val_logits = model(X_val_t)
    val_pred = torch.argmax(val_logits, dim=1)
    val_pred_np = val_pred.cpu().numpy()

# results: accuracy, confusion matrix, per class f1 table
val_acc = float((val_pred_np == y_val_0).mean())
class_ids = np.unique(y_train)
n_classes = len(class_ids)
cm = np.zeros((n_classes, n_classes), dtype=int)
for t, p in zip(y_val_0, val_pred_np):
    cm[int(t), int(p)] += 1
precision = np.zeros(n_classes, dtype=float)
recall = np.zeros(n_classes, dtype=float)
f1 = np.zeros(n_classes, dtype=float)
support = cm.sum(axis=1)
for i in range(n_classes):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    precision[i] = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall[i] = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    denom = precision[i] + recall[i]
    f1[i] = (2 * precision[i] * recall[i] / denom) if denom > 0 else 0.0
macro_f1 = float(f1.mean())
overlap = len(set(np.unique(subj_train[fit_mask])).intersection(set(np.unique(subj_train[val_mask]))))
print('Baseline 1 - MLP on flattened windows (T*C)')
print(f'Subject split seed: {split_seed}')
print(f'Fit subjects: {len(np.unique(subj_train[fit_mask]))}, windows: {X_fit.shape[0]}')
print(f'Val subjects: {len(np.unique(subj_train[val_mask]))}, windows: {X_val.shape[0]}')
print(f'Subject overlap (fit vs val): {overlap}')
print(f'Primary metric - Validation accuracy: {val_acc:.4f}')
print(f'Macro-F1 (reference): {macro_f1:.4f}')
print('\nConfusion matrix (rows=true, cols=pred; class ids 1..6):')
print(cm)
print('\nPer-class F1 table:')
print('class_id,activity,precision,recall,f1,support')
for idx, cid in enumerate(class_ids):
    print(f"{int(cid)},{activity_map.get(int(cid), 'UNKNOWN')},{precision[idx]:.4f},{recall[idx]:.4f},{f1[idx]:.4f},{int(support[idx])}")
assert overlap == 0, 'Overlap exists.'


Baseline 1 - MLP on flattened windows (T*C)
Subject split seed: 42
Fit subjects: 17, windows: 5800
Val subjects: 4, windows: 1552
Subject overlap (fit vs val): 0
Primary metric - Validation accuracy: 0.8750
Macro-F1 (reference): 0.8725

Confusion matrix (rows=true, cols=pred; class ids 1..6):
[[238   2   0  15   4   0]
 [ 19 181  10  18   5   0]
 [ 10   8 155  37   6   0]
 [  0   0   0 266   3   0]
 [  0   0   0  57 228   0]
 [  0   0   0   0   0 290]]

Per-class F1 table:
class_id,activity,precision,recall,f1,support
1,WALKING,0.8914,0.9189,0.9049,259
2,WALKING_UPSTAIRS,0.9476,0.7768,0.8538,233
3,WALKING_DOWNSTAIRS,0.9394,0.7176,0.8136,216
4,SITTING,0.6768,0.9888,0.8036,269
5,STANDING,0.9268,0.8000,0.8588,285
6,LAYING,1.0000,1.0000,1.0000,290
